# 04 — PyTorch Basics

**Topics covered:**
1. Tensors — creation, dtypes, devices
2. Tensor operations
3. Autograd — automatic differentiation
4. Building a simple neural network with `nn.Module`
5. Training loop — regression example
6. Training loop — classification example (Iris)
7. Saving & loading a model
8. Mini-exercises

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from ds_sandbox.dataset import load_iris

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {DEVICE}')

PyTorch version : 2.3.1
Device          : cpu


## 1. Tensors — Creation, dtypes, Devices

In [2]:
# From Python / NumPy
t1 = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
t2 = torch.from_numpy(np.array([5., 6., 7., 8.]))
print('t1:', t1, '| dtype:', t1.dtype)
print('t2:', t2, '| dtype:', t2.dtype)

# Factory functions
print(torch.zeros(2, 3))
print(torch.ones(2, 3))
print(torch.rand(2, 3))         # uniform [0, 1)
print(torch.randn(2, 3))        # standard normal
print(torch.arange(0, 10, 2))   # [0, 2, 4, 6, 8]

t1: tensor([1., 2., 3., 4.]) | dtype: torch.float32
t2: tensor([5., 6., 7., 8.], dtype=torch.float64) | dtype: torch.float64
tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[0.7052, 0.2323, 0.8172],
        [0.4842, 0.8827, 0.8607]])
tensor([[-1.6841,  1.2899,  0.4199],
        [-0.3570, -0.4107,  0.5084]])
tensor([0, 2, 4, 6, 8])


In [3]:
# Attributes
x = torch.rand(3, 4)
print('shape  :', x.shape)
print('ndim   :', x.ndim)
print('device :', x.device)
print('dtype  :', x.dtype)

# Move to device
x_dev = x.to(DEVICE)
print('on device:', x_dev.device)

shape  : torch.Size([3, 4])
ndim   : 2
device : cpu
dtype  : torch.float32
on device: cpu


## 2. Tensor Operations

In [4]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

print('a + b  :\n', a + b)
print('a * b  :\n', a * b)          # element-wise
print('a @ b  :\n', a @ b)          # matrix multiply
print('a.T    :\n', a.T)            # transpose
print('mean(a):', a.mean())
print('sum(a) :', a.sum())

# Reshaping
c = torch.arange(12).float()
print('\nReshaped 3x4:\n', c.reshape(3, 4))
print('Unsqueeze dim 0:', c.reshape(3, 4).unsqueeze(0).shape)

a + b  :
 tensor([[ 6.,  8.],
        [10., 12.]])
a * b  :
 tensor([[ 5., 12.],
        [21., 32.]])
a @ b  :
 tensor([[19., 22.],
        [43., 50.]])
a.T    :
 tensor([[1., 3.],
        [2., 4.]])
mean(a): tensor(2.5000)
sum(a) : tensor(10.)

Reshaped 3x4:
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
Unsqueeze dim 0: torch.Size([1, 3, 4])


## 3. Autograd — Automatic Differentiation

In [5]:
# Compute gradient of f(x) = x^3 + 2x^2 at x = 2
x = torch.tensor(2.0, requires_grad=True)
f = x**3 + 2*x**2
f.backward()   # df/dx = 3x^2 + 4x -> at x=2: 12 + 8 = 20
print(f'f(2)   = {f.item():.1f}')
print(f"f'(2)  = {x.grad.item():.1f}  (expected 20.0)")

f(2)   = 16.0
f'(2)  = 20.0  (expected 20.0)


In [6]:
# Stop gradient tracking with no_grad
with torch.no_grad():
    y = x * 3
print('y requires_grad:', y.requires_grad)  # False

y requires_grad: False


## 4. Building a Neural Network with `nn.Module`

In [7]:
class MLP(nn.Module):
    """A simple multi-layer perceptron."""
    def __init__(self, in_features: int, hidden: int, out_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_features),
        )

    def forward(self, x):
        return self.net(x)


model = MLP(in_features=4, hidden=16, out_features=1)
print(model)

# Count parameters
total = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total:,}')

MLP(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
  )
)

Total parameters: 369


## 5. Training Loop — Regression

In [8]:
torch.manual_seed(42)

# Generate synthetic y = 3x + 5 + noise
X = torch.linspace(-3, 3, 100).unsqueeze(1)
y = 3 * X + 5 + torch.randn_like(X) * 0.5

# Simple linear regression model
reg_model = nn.Linear(1, 1).to(DEVICE)
optimizer = optim.Adam(reg_model.parameters(), lr=0.05)
loss_fn   = nn.MSELoss()

X, y = X.to(DEVICE), y.to(DEVICE)

losses = []
for epoch in range(200):
    pred = reg_model(X)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

w = reg_model.weight.item()
b = reg_model.bias.item()
print(f'Learned: y = {w:.2f}x + {b:.2f}  (true: y = 3.00x + 5.00)')
print(f'Final loss: {losses[-1]:.4f}')

Learned: y = 2.99x + 4.76  (true: y = 3.00x + 5.00)
Final loss: 0.3189


In [9]:
# Plot loss curve
import plotly.express as px
fig = px.line(y=losses, labels={'index': 'Epoch', 'y': 'MSE Loss'},
              title='Regression Training Loss')
fig.show()

## 6. Training Loop — Classification (Iris)

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

iris = load_iris()
feature_cols = ['sepal length (cm)', 'sepal width (cm)',
                'petal length (cm)', 'petal width (cm)']

X_np = iris[feature_cols].values.astype(np.float32)
y_np = iris['target'].values.astype(np.int64)

# Normalise
scaler = StandardScaler()
X_np = scaler.fit_transform(X_np)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42, stratify=y_np
)

# Convert to tensors
Xtr = torch.from_numpy(X_train).to(DEVICE)
ytr = torch.from_numpy(y_train).to(DEVICE)
Xte = torch.from_numpy(X_test).to(DEVICE)
yte = torch.from_numpy(y_test).to(DEVICE)

print(f'Train: {Xtr.shape} | Test: {Xte.shape}')

Train: torch.Size([120, 4]) | Test: torch.Size([30, 4])


In [11]:
torch.manual_seed(42)

clf = MLP(in_features=4, hidden=32, out_features=3).to(DEVICE)
optimizer = optim.Adam(clf.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()

train_losses, val_accs = [], []

for epoch in range(200):
    clf.train()
    logits = clf(Xtr)
    loss   = loss_fn(logits, ytr)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    clf.eval()
    with torch.no_grad():
        preds = clf(Xte).argmax(dim=1)
        acc   = (preds == yte).float().mean().item()
    val_accs.append(acc)

print(f'Final val accuracy: {val_accs[-1]*100:.1f}%')

Final val accuracy: 96.7%


In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Cross-Entropy Loss', 'Validation Accuracy'])
fig.add_trace(go.Scatter(y=train_losses, mode='lines', name='Train Loss'), row=1, col=1)
fig.add_trace(go.Scatter(y=val_accs,    mode='lines', name='Val Acc'),    row=1, col=2)
fig.update_layout(title='Iris Classifier Training', showlegend=False)
fig.show()

## 7. Saving & Loading a Model

In [13]:
import os

os.makedirs('../models', exist_ok=True)
model_path = '../models/iris_mlp.pt'

# Save
torch.save(clf.state_dict(), model_path)
print(f'Model saved to {model_path}')

# Load into a fresh model
loaded = MLP(in_features=4, hidden=32, out_features=3).to(DEVICE)
loaded.load_state_dict(torch.load(model_path, map_location=DEVICE))
loaded.eval()

with torch.no_grad():
    check_acc = (loaded(Xte).argmax(1) == yte).float().mean().item()
print(f'Loaded model accuracy: {check_acc*100:.1f}%')

Model saved to ../models/iris_mlp.pt
Loaded model accuracy: 96.7%


## 8. Mini-Exercises

In [14]:
# Exercise 1: Compute the gradient of f(x,y) = x^2*y + y^3 at (x=1, y=2)
# Your code here


In [15]:
# Exercise 2: Add Dropout (p=0.2) and BatchNorm to the MLP and retrain on Iris
# Your code here


In [16]:
# Exercise 3: Use DataLoader with batch_size=16 to retrain the classifier
# Your code here
